# W9: 판매 이상감지(3σ)

매출 데이터에서 3σ 이상치를 탐지하고, 최소 2건의 이상치를 기본 데이터에 보장합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

rng = np.random.RandomState(42)
sales = pd.DataFrame({
    "일자": pd.date_range("2026-01-01", periods=30, freq="D"),
    "매출": rng.normal(120000, 10000, 30).astype(int)
})
# 반드시 2건 이상치 삽입
sales.loc[5, "매출"] = 40000
sales.loc[17, "매출"] = 250000

sales["이동평균"] = sales["매출"].rolling(7, min_periods=1).mean()
sales["이동표준편차"] = sales["매출"].rolling(7, min_periods=1).std(ddof=0).fillna(0)
sales["z"] = (sales["매출"] - sales["이동평균"]) / sales["이동표준편차"].replace(0, 1)

anomaly = sales[sales["z"].abs() > 3].copy()
print("이상치 건수:", len(anomaly))
print(anomaly[["일자", "매출", "이동평균", "z"]])

fig, ax = plt.subplots(figsize=(10,3))
ax.plot(sales["일자"], sales["매출"], label="매출")
ax.plot(sales["일자"], sales["이동평균"], label="MA(7)")
ax.scatter(anomaly["일자"], anomaly["매출"], c="red", label="anomaly")
ax.legend()
ax.set_title("3σ 이상매출 탐지")
plt.tight_layout()
plt.show()
